In [1]:
# Import necessary libraries
import json
import os
from typing import Any, Dict

import boto3
import requests
from dotenv import load_dotenv

load_dotenv()
print("We are up and running!")

We are up and running!


AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [2]:
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

Bedrock client ready — model: eu.amazon.nova-lite-v1:0


In [3]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

I'm unable to provide real-time data as my training only includes information up until 2022. To find the current price of Bitcoin, you can check reputable financial news websites, cryptocurrency exchanges, or use dedicated financial apps that track cryptocurrency prices. Some popular exchanges where you can check the current price include:

- Coinbase
- Binance
- Kraken
- CoinMarketCap
- CoinGecko

These platforms offer up-to-date prices and additional information about Bitcoin and other cryptocurrencies.


In [4]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "7058ddeb-3ace-423f-a993-ae8dad46e7b4",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 10:58:08 GMT",
      "content-type": "application/json",
      "content-length": "721",
      "connection": "keep-alive",
      "x-amzn-requestid": "7058ddeb-3ace-423f-a993-ae8dad46e7b4"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm unable to provide real-time data as my training only includes information up until 2022. To find the current price of Bitcoin, you can check reputable financial news websites, cryptocurrency exchanges, or use dedicated financial apps that track cryptocurrency prices. Some popular exchanges where you can check the current price include:\n\n- Coinbase\n- Binance\n- Kraken\n- CoinMarketCap\n- CoinGecko\n\nThese platforms offer up-to-date prices and additional information about Bitcoin and other cryptocu

In [5]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

{'symbol': 'BTCUSDT', 'price': '66250.00000000'}


In [6]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [7]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

BTC Price in USDT: 66249.99


In [8]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [9]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "e7a40df7-2925-4c58-ae75-1323ecb08b2b",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 10:59:37 GMT",
      "content-type": "application/json",
      "content-length": "554",
      "connection": "keep-alive",
      "x-amzn-requestid": "e7a40df7-2925-4c58-ae75-1323ecb08b2b"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>The User wants to know the current price of Bitcoin. I have a tool that can fetch the current price of cryptocurrencies in USDT from Binance. I need to call this tool with the symbol for Bitcoin, which is BTCUSDT.</thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_rElKL8GWO62zpwzNTDoTSK",
            "name": "get_crypto_price",
            "input": {
              "symbol": "BTCUSDT"
            }
          }
        }
      ]
    }
  },
  "stopReason": 

In [10]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: get_crypto_price
Tool input:     {'symbol': 'BTCUSDT'}
Tool use ID:    tooluse_rElKL8GWO62zpwzNTDoTSK


In [11]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

Result from function: 66259.63


In [12]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "09b267c2-4573-47aa-b098-c19781780588",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 10:59:46 GMT",
      "content-type": "application/json",
      "content-length": "413",
      "connection": "keep-alive",
      "x-amzn-requestid": "09b267c2-4573-47aa-b098-c19781780588"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>I have used the tool to get the current price of Bitcoin, and the result is 66259.63 USDT. I can now provide this information to the User.</thinking>\n\nThe current price of Bitcoin is 66259.63 USDT."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 554,
    "outputTokens": 62,
    "totalTokens": 616
  },
  "metrics": {
    "latencyMs": 637
  }
}


In [13]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

The current price of Bitcoin is 66259.63 USDT.
